# Assignment 5 — Annual Report Fanciness (Google Colab GPU)

This notebook runs the **same version-controlled classifier in GitHub**, using a Colab session instead of your laptop.

**Before running:** In Colab choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). Free GPU access is not guaranteed and Colab sessions may end. Don't upload SDU's licensed Orbis financial export here; this notebook needs only publicly available annual-report PDFs and a local company manifest.

**Important:** First test just **one verified PDF**. A link returning 403 in Windows may also fail in Colab. Upload the PDF manually if necessary. Keep report URLs and years verified against official company sources. No scores or findings are fabricated.

In [ ]:
import subprocess
gpu=subprocess.run(["nvidia-smi"],capture_output=True,text=True)
if gpu.returncode != 0:
    raise RuntimeError("No NVIDIA GPU detected. In Colab choose Runtime > Change runtime type > T4 GPU, if available.")
print(gpu.stdout[:3000])

## 1. Get the current scripts from GitHub

In [ ]:
import os, pathlib, subprocess
ROOT=pathlib.Path("/content/annual-report-fanciness-analysis")
if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/MVEST22/annual-report-fanciness-analysis.git",str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
subprocess.run(["python","-m","pip","-q","install","-r","requirements.txt"],check=True)
(ROOT/"reports").mkdir(exist_ok=True)
print("Project ready:",ROOT)

## 2. Install and start Ollama inside this Colab session

This runs **Gemma 3 locally within the Colab VM**; it does not require a paid API key. Model downloads consume session time and storage. The model still needs a suitable GPU to perform well.

In [ ]:
import os, subprocess, time, requests, pathlib
# Official Ollama Linux installer. Review the installation script if your security policy requires it.
subprocess.run(["apt-get","update","-qq"],check=True)
subprocess.run(["apt-get","install","-y","zstd"],check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",shell=True,check=True,executable="/bin/bash")
log=open("/content/ollama-server.log","a")
server=subprocess.Popen(["ollama","serve"],stdout=log,stderr=subprocess.STDOUT,env=os.environ.copy())
for i in range(40):
    try:
        if requests.get("http://127.0.0.1:11434/api/tags",timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not become ready; inspect /content/ollama-server.log")
subprocess.run(["ollama","pull","gemma3:4b"],check=True)
print("Ollama model ready")

## 3. Upload your report manifest

Upload a CSV containing `company_name,bvd_id,report_url,report_year`, such as `report_manifest.csv`. The uploaded file is stored only in this temporary session. **Do not upload licensed Orbis profit data.**

In [ ]:
from google.colab import files
uploaded=files.upload()
csv_names=[n for n in uploaded if n.lower().endswith(".csv")]
if len(csv_names)!=1: raise ValueError("Upload exactly one manifest CSV in this step.")
manifest_path=ROOT/"report_manifest.csv"
manifest_path.write_bytes(uploaded[csv_names[0]])
import csv
with manifest_path.open(newline="",encoding="utf-8-sig") as f:
    rows=list(csv.DictReader(f))
assert rows and {"company_name","bvd_id","report_url","report_year"}.issubset(rows[0]), "Manifest is missing required columns."
print("Manifest records:",len(rows))
for i,r in enumerate(rows[:10]): print(i+1,r["company_name"],r["bvd_id"])

## 4. Upload one manually verified annual-report PDF

Choose a report from the manifest above and enter its **row number**. Upload the PDF from your own device; it will be stored under `reports/<bvd_id>.pdf`. This avoids 403-blocked direct links. **Check that the PDF belongs to that company and is its latest annual report.**

In [ ]:
from google.colab import files
selection=int(input("Which manifest row number (1-based) is this PDF for? "))
if not 1 <= selection <= len(rows): raise ValueError("Row number outside manifest.")
company=rows[selection-1]
selected_id=company["bvd_id"]
print("Selected:",company["company_name"],selected_id)
upload_pdf=files.upload()
pdf_files=[n for n in upload_pdf if n.lower().endswith(".pdf")]
if len(pdf_files)!=1: raise ValueError("Upload exactly one PDF.")
import re
safe_id=re.sub(r"[^a-zA-Z0-9_-]","_",selected_id)
target=ROOT/"reports"/(safe_id+".pdf")
pdfdata=upload_pdf[pdf_files[0]]
if not pdfdata.startswith(b"%PDF"): raise ValueError("The uploaded file does not appear to be a PDF.")
target.write_bytes(pdfdata)
with (ROOT/"test_manifest.csv").open("w",newline="",encoding="utf-8") as f:
    writer=csv.DictWriter(f,fieldnames=["company_name","bvd_id","report_url","report_year"])
    writer.writeheader();writer.writerow({k:company.get(k,"") for k in writer.fieldnames})
print("Ready to classify:",target.name,"(",len(pdfdata),"bytes)")

## 5. Run the one-report test

The classifier asks Gemma 3 to define 'fancy' and shows the definition before scoring. **Read and approve the definition.** This cell sends `YES` to the CLI; only run it if you have already reviewed and approved the same model/rubric in the prior local test. Otherwise, run the model-definition check separately first (next cell). The final five-dimension scoring uses the regular two-page method, not the single-cover pilot.

In [ ]:
from pathlib import Path
import requests
from IPython.display import display, Markdown
from classify_reports import RUBRIC
response=requests.post("http://127.0.0.1:11434/api/chat",
 json={"model":"gemma3:4b","stream":False,"messages":[{"role":"user","content":"Define your visual fanciness scale from 1 to 5 for corporate annual reports, without using profit or firm size. Compare it with: "+RUBRIC}],"options":{"temperature":0,"num_predict":450}},timeout=240)
response.raise_for_status()
definition=response.json()["message"]["content"]
print(definition)
(ROOT/"fanciness_definition.txt").write_text(definition+"\n\nResearcher rubric:\n"+RUBRIC,encoding="utf-8")
approval=input("Having reviewed the definition above, type YES to approve: ")
if approval.strip()!="YES": raise RuntimeError("Not approved; stop before classification.")
print("Definition approved for this session.")

In [ ]:
# Run one selected report with the previously approved definition.
import subprocess, sys
result=subprocess.run([sys.executable,"classify_reports.py","--manifest","test_manifest.csv",
                       "--limit","1","--reuse-definition"],input="YES\n",text=True,
                      cwd=str(ROOT),timeout=900)
if result.returncode: raise RuntimeError("Classifier returned an error.")
import pandas as pd
scores=pd.read_csv(ROOT/"scores.csv")
display(scores[["company_name","fanciness","weighted_score","status","rationale"]])
if not (scores["status"]=="scored").all():
    print("Not successfully scored: inspect the status and PDF contents before proceeding.")

## 6. Download the test results before leaving Colab

Colab VM files are temporary. Download `scores.csv` and `fanciness_definition.txt` to your computer, or deliberately save approved results to your own secure storage. **Do not overwrite your previous score files without backing them up.**

In [ ]:
from google.colab import files
for name in ["scores.csv","fanciness_definition.txt"]:
    path=ROOT/name
    if path.exists(): files.download(str(path))
print("Keep the downloaded files. The public GitHub repository stores the code, not the private Orbis dataset.")

## After the pilot

For larger-scale classification, first collect and validate the PDF manifest for each company. Upload report PDFs or a ZIP containing a `reports/` folder into the Colab session, then use the existing `classify_reports.py` script (without `--fast`) on the verified complete manifest. Split work into batches and export results after each batch. Missing PDFs, year mismatches and blocked links must remain flagged—not assigned invented scores. The regression will be executed locally with your licensed Orbis profit figures using `regression.py`.

## 7. Batch processing with checkpoints (optional, after the one-report test)

Run these cells after the pilot succeeds. **The five-company manifest is only a pilot.** For all companies, upload a newly checked, full company manifest. The classifier cannot automatically turn search candidates into verified annual reports. Keep Orbis profit figures out of Colab. All reports must be latest available complete annual reports, checked against the issuer.

You can upload a ZIP containing multiple PDFs, named exactly `<bvd_id>.pdf` (e.g. `DK10403782.pdf`). Reusing the PDF naming convention eliminates manual company selection. The scoring script saves `batch_scores.csv` after every attempt and resumes without re-scoring completed companies. A new Colab session requires re-uploading the manifest, PDFs, and any previously saved `batch_scores.csv`.

In [ ]:
import subprocess, os
os.chdir(ROOT)
subprocess.run(["git","pull","--ff-only"],check=True)
print("Fetched latest batch_classify.py and updated page sampling.")

### Upload multiple PDFs at once

Create a ZIP on your laptop containing downloaded annual-report PDFs named by their BvD identifiers. The PDF paths can be inside subfolders. This upload rejects files that are not valid PDF bytes, large individual PDFs, and names that aren't present in the manifest.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import io, re
archive=files.upload()
zips=[n for n in archive if n.lower().endswith(".zip")]
if len(zips)!=1: raise ValueError("Upload exactly one ZIP containing PDFs.")
allowed={re.sub(r'[^a-zA-Z0-9_-]','_',r['bvd_id']) for r in rows}
count=0
with ZipFile(io.BytesIO(archive[zips[0]])) as z:
    for info in z.infolist():
        filename=Path(info.filename).name
        if not filename.lower().endswith(".pdf") or info.is_dir(): continue
        stem=filename[:-4]
        if stem not in allowed or info.file_size>80_000_000: continue
        data=z.read(info)
        if not data.startswith(b"%PDF"): continue
        (ROOT/"reports"/(stem+".pdf")).write_bytes(data)
        count+=1
print("Matching valid PDFs extracted:",count,"of",len(allowed),"manifest companies")

### Resume any previously downloaded scores (optional)

If Colab restarted, run the following cell and upload your earlier `batch_scores.csv`. If you haven't run a batch before, skip it. The first Vestas pilot was saved as `scores.csv`; it is intentionally **not reused** because the updated full-method classifier samples more pages.

In [ ]:
from google.colab import files
import pathlib
print("OPTIONAL: Select your prior batch_scores.csv or cancel this cell if no prior batch exists.")
resume=files.upload()
if resume:
    names=[n for n in resume if n=="batch_scores.csv"]
    if len(names)!=1: raise ValueError("Only upload batch_scores.csv here.")
    (ROOT/"batch_scores.csv").write_bytes(resume[names[0]])
    print("Previous batch restored")

### Run a small batch

The current `report_manifest.csv` is used; only matching locally uploaded PDFs are scored. Reports without a PDF remain unscored. Review the output after a small batch before running the full sample. The page-selection method includes cover, early, middle and later pages; still manually audit a subset for quality.

In [ ]:
import subprocess, sys
result=subprocess.run([sys.executable,"batch_classify.py","--manifest","report_manifest.csv",
                       "--output","batch_scores.csv","--limit","5"],cwd=str(ROOT))
print("Process exit code:",result.returncode)
import pandas as pd
batch=pd.read_csv(ROOT/"batch_scores.csv")
display(batch[["company_name","fanciness","status","pages_reviewed"]])
from google.colab import files
files.download(str(ROOT/"batch_scores.csv"))

### Continue after verifying the first batch

When you have more verified PDFs, upload another ZIP and run the next cell. Existing successful scores are skipped. If files were missing in an earlier pass, use `--retry-errors` to attempt them after uploading PDFs. Download `batch_scores.csv` after each batch.

In [ ]:
import subprocess, sys
result=subprocess.run([sys.executable,"batch_classify.py","--manifest","report_manifest.csv",
                       "--output","batch_scores.csv","--retry-errors","--limit","15"],cwd=str(ROOT))
print("Exit code:",result.returncode)
from google.colab import files
files.download(str(ROOT/"batch_scores.csv"))

## 8. Bulk report discovery and download (optional, separate from scoring)

**Start here for scaling beyond the five-company pilot.** Upload the previously prepared 139-company `report_manifest.csv`, which contains *company names and BvD IDs only*. This file is safe to use only if it has no licensed financial data. Keep all profit data on your computer.

Use `discover_reports.py` to generate public search *candidates*. It does **not** verify that a PDF belongs to the right issuer, is the latest annual report, or that a company is currently listed. Review each result and fill `report_url` and `report_year` in your manifest using the issuer's official annual-report website. Do **not** automatically classify unreviewed search hits.

The downloader only uses those reviewed manifest URLs. It checks file signatures and PDF readability, saves successes, records errors such as 403, and skips missing links. Save results and report PDFs outside the temporary Colab runtime.

In [ ]:
# Upload the full company manifest; this replaces the five-company pilot file.
from google.colab import files
import csv
upload=files.upload()
names=[n for n in upload if n.lower().endswith(".csv")]
if len(names)!=1: raise ValueError("Select one CSV manifest.")
full_manifest=ROOT/"report_manifest.csv"
full_manifest.write_bytes(upload[names[0]])
with full_manifest.open(newline="",encoding="utf-8-sig") as f:
    full_rows=list(csv.DictReader(f))
required={"company_name","bvd_id","report_url","report_year"}
if not full_rows or not required.issubset(full_rows[0]): raise ValueError("Wrong manifest columns")
if any(k.lower() in full_rows[0] for k in ("profit","net_income","profit_m_dkk")):
    raise ValueError("Do not upload licensed Orbis financial fields to Colab")
print("Companies:",len(full_rows),"Approved report URLs:",sum(bool(r["report_url"].strip()) for r in full_rows))

In [ ]:
# Discover candidate public PDFs. Start with 5 companies as a test.
# Search results are ONLY leads. Do not treat them as approved reports.
import subprocess,sys,os
os.chdir(ROOT)
subprocess.run(["git","pull","--ff-only"],check=True)
subprocess.run([sys.executable,"discover_reports.py","--manifest",str(full_manifest),
                "--out","report_candidates.csv","--limit","5"],check=True)
import pandas as pd
candidates=pd.read_csv("report_candidates.csv")
display(candidates.head(10))
from google.colab import files
files.download("report_candidates.csv")

### After reviewing candidates

On your computer, verify candidates against each company's official investor-relations page and update the manifest's `report_url` and `report_year`. Re-upload the **reviewed** manifest using the cell below. For the full sample, run `discover_reports.py` without `--limit 5` in a separate step. Some issuers block automated PDFs (HTTP 403), so manually download those and include them in your PDF ZIP. Do not mislabel a missing report as a low fanciness score.

In [ ]:
# Upload a reviewed manifest with approved report URLs and reporting years.
from google.colab import files
u=files.upload()
names=[n for n in u if n.lower().endswith(".csv")]
if len(names)!=1: raise ValueError("Select only the reviewed manifest CSV.")
full_manifest.write_bytes(u[names[0]])
with full_manifest.open(newline="",encoding="utf-8-sig") as f:
    approved_rows=list(csv.DictReader(f))
if not approved_rows or not {"company_name","bvd_id","report_url","report_year"}.issubset(approved_rows[0]):
    raise ValueError("Manifest missing fields")
if any(k.lower() in approved_rows[0] for k in ("profit","net_income","profit_m_dkk")):
    raise ValueError("Private profit data must not enter Colab")
print("Reviewed URL count:",sum(bool(r["report_url"].strip()) for r in approved_rows))

In [ ]:
# Bulk-download ONLY reviewed URLs, starting with up to five rows.
import subprocess,sys
subprocess.run([sys.executable,"download_reports.py","--manifest",str(full_manifest),
                "--out","download_log.csv","--limit","5"],check=True,cwd=str(ROOT))
import pandas as pd
display(pd.read_csv(ROOT/"download_log.csv"))
from google.colab import files
files.download(str(ROOT/"download_log.csv"))

### Download the report archive and complete collection

After validating the five-row test, remove `--limit 5` from the downloader invocation to attempt all reviewed links. For any 403 blocks, download the PDF manually from the issuer's site and name it `<bvd_id>.pdf`; upload it as a ZIP in Section 7. Run the batch classifier only after confirming report years and current listing status. The ZIP below contains publicly available report PDFs already downloaded to the Colab runtime. Check copyright and permitted use before sharing PDFs publicly; do not commit them to GitHub.

In [ ]:
# Create a ZIP of downloaded report PDFs for local backup, not for GitHub publication.
from zipfile import ZipFile,ZIP_DEFLATED
archive=ROOT/"public_report_pdfs.zip"
with ZipFile(archive,"w",ZIP_DEFLATED) as z:
    for p in sorted((ROOT/"reports").glob("*.pdf")):
        z.write(p,arcname="reports/"+p.name)
print("PDF files:",len(list((ROOT/"reports").glob("*.pdf"))))
from google.colab import files
files.download(str(archive))